# Abdelrahman Mostafa — ViT-B/16 Baseline

**DSAI 305 — Phase 2 | Team 505**  
**Task:** Binary Pneumonia Classification on NIH ChestX-ray14

---

## 1 — Paper / Model Reference

| Item | Detail |
|------|--------|
| Architecture | ViT-B/16 (Dosovitskiy et al., *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*, ICLR 2021) |
| Pretrained weights | ImageNet-1K (torchvision) |
| CXR reference | Singh et al., *Assessment of Chest X-ray Pathology Detection Using Vision Transformers*, 2022 |

Singh et al. evaluated Vision Transformers for chest X-ray pathology detection and demonstrated that ViT-based models can achieve competitive or superior performance compared to CNN baselines. ViT-B/16 applies the Transformer architecture directly to 16×16 image patches, using self-attention instead of convolutions.

**Why this model is in the unified comparison:** ViT-B/16 represents the Transformer paradigm — a fundamentally different architecture from all CNN baselines in this project. Including it tests whether attention-based models outperform convolutional approaches for chest X-ray pneumonia detection, as investigated by Singh et al.


## 2 — Objective

Train a **preliminary ViT-B/16 baseline** for binary pneumonia detection:

- Fine-tune ImageNet-pretrained ViT-B/16 on NIH ChestX-ray14 pneumonia labels
- Handle severe class imbalance (~1.3% positive rate) with weighted BCE loss
- Evaluate with accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrix
- Use `train_dev` for quick debugging; switch to full `train.csv + val.csv` for final results

**This is a preliminary baseline.** XAI analysis will follow in Phase 3.

### Shared Project Base

This notebook is part of the **Team 505 unified pipeline**. All four team members share:

| Component | Source |
|---|---|
| Preprocessing | `Team505_Preprocessing.ipynb` |
| Exploratory Data Analysis | `Team505_EDA.ipynb` |
| Training split | `data/splits/train.csv` |
| Validation split | `data/splits/val.csv` |
| Test split (locked) | `data/splits/test.csv` |
| Dev subset | `data/splits/train_dev.csv` |

**Consistency guarantees across all notebooks:**
- Same NIH ChestX-ray14 dataset
- Same patient-wise split logic (no patient leakage)
- Same binary pneumonia target (`target_pneumonia`)
- Same evaluation metric family (ROC-AUC, F1, precision, recall, accuracy, confusion matrix)
- Same output-saving pattern (`outputs/<member_name>/`)


---
## 3 — Imports & Configuration

In [1]:
import sys
print(sys.executable)

c:\Users\s-amm\Downloads\ChestX-ray14\Team505_phase2\.venv\Scripts\python.exe


In [2]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings('ignore')

# ---- Device detection ----
CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device('cuda' if CUDA_AVAILABLE else 'cpu')

# ---- Reproducibility ----
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if CUDA_AVAILABLE:
    torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = CUDA_AVAILABLE

print(f'PyTorch version   : {torch.__version__}')
print(f'CUDA build        : {torch.version.cuda}')
print(f'CUDA available    : {CUDA_AVAILABLE}')
print(f'Device selected   : {DEVICE}')
if CUDA_AVAILABLE:
    print(f'GPU name          : {torch.cuda.get_device_name(0)}')
    print(f'GPU memory        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('GPU name          : N/A (running on CPU)')

PyTorch version   : 2.11.0+cu126
CUDA build        : 12.6
CUDA available    : True
Device selected   : cuda
GPU name          : NVIDIA GeForce RTX 3070 Laptop GPU
GPU memory        : 8.6 GB


In [3]:
# ==============================================================================
# RUN MODE CONFIGURATION
# ==============================================================================
# "smoke" = tiny subset (512 samples), 1 epoch  — verify notebook works (~15s)
# "dev"   = full train_dev.csv, 5 epochs         — preliminary results
# "full"  = official train.csv + val.csv, 15 ep   — final results (switch later)
# ==============================================================================

RUN_MODE = "dev"   # <-- "smoke", "dev", or "full"

# ---- Derived settings ----
USE_DEV_MODE  = (RUN_MODE != "full")
SMOKE_SAMPLES = 512

# ---- Hyperparameters ----
IMG_SIZE      = 224
BATCH_SIZE    = 16 if CUDA_AVAILABLE else 16
LEARNING_RATE = 3e-5
WEIGHT_DECAY  = 1e-5

# Epoch strategy (early stopping may terminate sooner)
_EPOCH_MAP = {"smoke": 1, "dev": 5, "full": 15}
NUM_EPOCHS = _EPOCH_MAP[RUN_MODE]

# ---- Early stopping ----
EARLY_STOP_PATIENCE = 3
EARLY_STOP_ENABLED  = True

# ---- DataLoader settings ----
NUM_WORKERS        = 0    # Windows/Jupyter: num_workers=0 avoids spawn overhead
PIN_MEMORY         = CUDA_AVAILABLE
PERSISTENT_WORKERS = False

# ---- Paths ----
PROJECT_ROOT = Path('..').resolve()
DATA_SPLITS  = PROJECT_ROOT / 'data' / 'splits'
OUTPUT_DIR   = PROJECT_ROOT / 'outputs' / 'Abdelrahman_Mostafa'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Run mode       : {RUN_MODE.upper()}')
print(f'Epochs (max)   : {NUM_EPOCHS}')
print(f'Early stopping : patience={EARLY_STOP_PATIENCE} ({"ON" if EARLY_STOP_ENABLED else "OFF"})')
print(f'Batch size     : {BATCH_SIZE}')
print(f'Num workers    : {NUM_WORKERS}')
print(f'Pin memory     : {PIN_MEMORY}')
print(f'Learning rate  : {LEARNING_RATE}')
print(f'Image size     : {IMG_SIZE}x{IMG_SIZE}')
print(f'Output dir     : {OUTPUT_DIR}')

Run mode       : DEV
Epochs (max)   : 5
Early stopping : patience=3 (ON)
Batch size     : 16
Num workers    : 0
Pin memory     : True
Learning rate  : 3e-05
Image size     : 224x224
Output dir     : C:\Users\s-amm\Downloads\ChestX-ray14\Team505_phase2\outputs\Abdelrahman_Mostafa


---
## 4 — Load Split CSVs

In [4]:
if RUN_MODE == "full":
    df_train = pd.read_csv(DATA_SPLITS / 'train.csv')
    df_val   = pd.read_csv(DATA_SPLITS / 'val.csv')
    print('[FULL MODE] Using train.csv + val.csv')
else:
    df_full_dev = pd.read_csv(DATA_SPLITS / 'train_dev.csv')
    
    all_pids = df_full_dev['patient_id'].unique()
    rng = np.random.RandomState(RANDOM_SEED)
    rng.shuffle(all_pids)
    split_idx = int(len(all_pids) * 0.8)
    train_pids = set(all_pids[:split_idx])
    val_pids   = set(all_pids[split_idx:])
    
    df_train = df_full_dev[df_full_dev['patient_id'].isin(train_pids)].copy()
    df_val   = df_full_dev[df_full_dev['patient_id'].isin(val_pids)].copy()
    
    if RUN_MODE == "smoke":
        df_train = df_train.head(SMOKE_SAMPLES)
        df_val   = df_val.head(SMOKE_SAMPLES // 4)
        print(f'[SMOKE MODE] Subset to {len(df_train)} train + {len(df_val)} val samples')
    else:
        print('[DEV MODE] Using train_dev.csv with internal 80/20 patient-wise split')

df_test = pd.read_csv(DATA_SPLITS / 'test.csv')

print(f'\nTrain : {len(df_train):>7,} images, {df_train["patient_id"].nunique():>6,} patients')
print(f'Val   : {len(df_val):>7,} images, {df_val["patient_id"].nunique():>6,} patients')
print(f'Test  : {len(df_test):>7,} images, {df_test["patient_id"].nunique():>6,} patients (locked)')

[DEV MODE] Using train_dev.csv with internal 80/20 patient-wise split

Train :   6,365 images,    469 patients
Val   :   1,720 images,    118 patients
Test  :  25,596 images,  2,797 patients (locked)


---
## 5 — Class Counts

In [5]:
for name, sdf in [('Train', df_train), ('Val', df_val)]:
    pos = int(sdf['target_pneumonia'].sum())
    neg = len(sdf) - pos
    rate = pos / len(sdf) * 100
    print(f'{name:5s}: {len(sdf):>7,} images  |  Pos: {pos:>5,}  Neg: {neg:>7,}  |  Pos rate: {rate:.2f}%')

n_pos_train = int(df_train['target_pneumonia'].sum())
n_neg_train = len(df_train) - n_pos_train
pos_weight_value = n_neg_train / max(n_pos_train, 1)
print(f'\npos_weight = {pos_weight_value:.2f}  (neg/pos ratio in training set)')

Train:   6,365 images  |  Pos:   588  Neg:   5,777  |  Pos rate: 9.24%
Val  :   1,720 images  |  Pos:   154  Neg:   1,566  |  Pos rate: 8.95%

pos_weight = 9.82  (neg/pos ratio in training set)


The class imbalance is severe. We use `BCEWithLogitsLoss` with `pos_weight` to upweight pneumonia-positive examples.

---
## 6 — Transforms

In [6]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Transforms defined:')
print(f'  Train: Resize({IMG_SIZE}) -> HFlip -> Rotate(±10°) -> ColorJitter -> Normalize')
print(f'  Val  : Resize({IMG_SIZE}) -> Normalize')

Transforms defined:
  Train: Resize(224) -> HFlip -> Rotate(±10°) -> ColorJitter -> Normalize
  Val  : Resize(224) -> Normalize


Mild, medically defensible augmentations for training only. ImageNet normalization is applied since the model was pretrained on ImageNet.

---
## 7 — Dataset & DataLoader

In [7]:
class ChestXrayDataset(Dataset):
    """
    PyTorch Dataset for NIH ChestX-ray14 binary pneumonia classification.
    
    - Loads grayscale PNG images and converts to 3-channel RGB
    - Handles image loading errors gracefully (returns black image)
    """
    
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.image_paths = self.df['image_path'].tolist()
        self.labels = self.df['target_pneumonia'].values.astype(np.float32)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (0, 0, 0))
        
        if self.transform:
            img = self.transform(img)
        
        return img, label


train_dataset = ChestXrayDataset(df_train, transform=train_transforms)
val_dataset   = ChestXrayDataset(df_val,   transform=val_transforms)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

print(f'Train: {len(train_dataset):,} samples -> {len(train_loader):,} batches')
print(f'Val  : {len(val_dataset):,} samples -> {len(val_loader):,} batches')

Train: 6,365 samples -> 398 batches
Val  : 1,720 samples -> 108 batches


Grayscale PNGs are converted to 3-channel RGB via `Image.convert('RGB')`. Failed image loads return a black placeholder.

---
## 8–9 — Load ViT-B/16 & Replace Classifier

In [8]:
model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)

# Replace classification head: ViT-B/16 features -> 1 output logit (binary)
num_features = model.heads.head.in_features
model.heads.head = nn.Linear(num_features, 1)

model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model         : ViT-B/16 (ImageNet pretrained)')
print(f'Classifier    : Linear({num_features} -> 1)')
print(f'Total params  : {total_params:,}')
print(f'Trainable     : {trainable:,}')
print(f'On device     : {DEVICE}')
print(f'Param device  : {next(model.parameters()).device}')

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to C:\Users\s-amm/.cache\torch\hub\checkpoints\vit_b_16-c867db91.pth


100%|██████████| 330M/330M [01:50<00:00, 3.14MB/s] 


Model         : ViT-B/16 (ImageNet pretrained)
Classifier    : Linear(768 -> 1)
Total params  : 85,799,425
Trainable     : 85,799,425
On device     : cuda
Param device  : cuda:0


ViT-B/16 is a Transformer-based architecture (~86M params) that is significantly heavier than CNNs. We replace the 1000-class ImageNet classification head with a single-output linear layer. ViT requires more memory; batch size may need to be reduced on laptops.

---
## 10–11 — Loss, Optimizer & Scheduler

In [9]:
pos_weight_tensor = torch.tensor([pos_weight_value], dtype=torch.float32).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

print(f'Loss      : BCEWithLogitsLoss (pos_weight={pos_weight_value:.2f})')
print(f'Optimizer : Adam (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})')
print(f'Scheduler : ReduceLROnPlateau (factor=0.5, patience=2)')

Loss      : BCEWithLogitsLoss (pos_weight=9.82)
Optimizer : Adam (lr=3e-05, wd=1e-05)
Scheduler : ReduceLROnPlateau (factor=0.5, patience=2)


---
## 12–14 — Training & Validation Loop

In [10]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch. Returns average loss."""
    model.train()
    running_loss = 0.0
    n_batches = 0
    
    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).unsqueeze(1)
        
        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        n_batches += 1
        
        if batch_idx == 0 and not hasattr(train_one_epoch, '_checked'):
            train_one_epoch._checked = True
            print(f'  [GPU check] images={images.device}, labels={labels.device}')
    
    return running_loss / max(n_batches, 1)


def validate(model, loader, criterion, device):
    """Validate. Returns loss, predictions, probabilities, labels."""
    model.eval()
    running_loss = 0.0
    n_batches = 0
    all_labels = []
    all_probs  = []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).unsqueeze(1)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            n_batches += 1
            
            probs = torch.sigmoid(outputs).cpu().numpy().flatten()
            all_probs.extend(probs)
            all_labels.extend(labels.cpu().numpy().flatten())
    
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)
    all_preds  = (all_probs >= 0.5).astype(int)
    
    return running_loss / max(n_batches, 1), all_preds, all_probs, all_labels

print('Training and validation functions defined.')

Training and validation functions defined.


In [11]:
print('=' * 70)
print(f'Starting training: up to {NUM_EPOCHS} epoch(s)  |  Mode: {RUN_MODE.upper()}')
print(f'Device: {DEVICE}  |  GPU: {CUDA_AVAILABLE}  |  Early stop: {EARLY_STOP_ENABLED}')
print('=' * 70)

history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_f1': []}
best_val_auc = 0.0
best_epoch = 0
epochs_no_improve = 0
best_model_path = OUTPUT_DIR / 'best_model.pth'

start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()
    
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    
    val_loss, val_preds, val_probs, val_labels = validate(
        model, val_loader, criterion, DEVICE
    )
    
    val_auc = roc_auc_score(val_labels, val_probs) if val_labels.sum() > 0 else 0.0
    val_f1  = f1_score(val_labels, val_preds, zero_division=0)
    val_acc = accuracy_score(val_labels, val_preds)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    history['val_f1'].append(val_f1)
    
    scheduler.step(val_loss)
    
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_epoch = epoch
        epochs_no_improve = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_auc': val_auc,
            'val_f1': val_f1,
            'val_loss': val_loss,
        }, best_model_path)
    else:
        epochs_no_improve += 1
    
    elapsed = time.time() - epoch_start
    lr_now = optimizer.param_groups[0]['lr']
    marker = ' *' if epoch == best_epoch else ''
    
    print(
        f'Epoch {epoch:2d}/{NUM_EPOCHS} | '
        f'Train: {train_loss:.4f} | '
        f'Val: {val_loss:.4f} | '
        f'AUC: {val_auc:.4f} | '
        f'F1: {val_f1:.4f} | '
        f'Acc: {val_acc:.4f} | '
        f'LR: {lr_now:.1e} | '
        f'{elapsed:.0f}s{marker}'
    )
    
    if EARLY_STOP_ENABLED and epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f'\nEarly stopping: no AUC improvement for {EARLY_STOP_PATIENCE} epochs.')
        break

actual_epochs = len(history['train_loss'])
total_time = time.time() - start_time
print(f'\nTraining done: {actual_epochs} epochs in {total_time/60:.1f} min.')
print(f'Best epoch: {best_epoch}  |  Best Val AUC: {best_val_auc:.4f}')
print(f'Model saved: {best_model_path}')

Starting training: up to 5 epoch(s)  |  Mode: DEV
Device: cuda  |  GPU: True  |  Early stop: True
  [GPU check] images=cuda:0, labels=cuda:0
Epoch  1/5 | Train: 1.2868 | Val: 1.4391 | AUC: 0.5403 | F1: 0.1644 | Acc: 0.0895 | LR: 3.0e-05 | 321s *
Epoch  2/5 | Train: 1.2604 | Val: 1.2530 | AUC: 0.5412 | F1: 0.1667 | Acc: 0.1919 | LR: 3.0e-05 | 307s *
Epoch  3/5 | Train: 1.2496 | Val: 1.2611 | AUC: 0.5543 | F1: 0.1648 | Acc: 0.0924 | LR: 3.0e-05 | 298s *
Epoch  4/5 | Train: 1.2473 | Val: 1.3016 | AUC: 0.5652 | F1: 0.1658 | Acc: 0.1227 | LR: 3.0e-05 | 306s *
Epoch  5/5 | Train: 1.2438 | Val: 1.2284 | AUC: 0.5770 | F1: 0.1849 | Acc: 0.4977 | LR: 3.0e-05 | 371s *

Training done: 5 epochs in 26.7 min.
Best epoch: 5  |  Best Val AUC: 0.5770
Model saved: C:\Users\s-amm\Downloads\ChestX-ray14\Team505_phase2\outputs\Abdelrahman_Mostafa\best_model.pth


---
## 15 — Training Curves

In [12]:
actual_epochs_trained = len(history['train_loss'])

if actual_epochs_trained >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    epochs_range = range(1, actual_epochs_trained + 1)

    axes[0].plot(epochs_range, history['train_loss'], 'o-', label='Train', color='#3498db')
    axes[0].plot(epochs_range, history['val_loss'], 'o-', label='Val', color='#e74c3c')
    if best_epoch > 0:
        axes[0].axvline(best_epoch, color='green', ls='--', alpha=0.5, label=f'Best (ep {best_epoch})')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss Curves', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs_range, history['val_auc'], 's-', color='#2ecc71')
    if best_epoch > 0:
        axes[1].axvline(best_epoch, color='green', ls='--', alpha=0.5)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('ROC-AUC')
    axes[1].set_title('Validation AUC', fontweight='bold'); axes[1].grid(alpha=0.3)

    axes[2].plot(epochs_range, history['val_f1'], 'd-', color='#9b59b6')
    if best_epoch > 0:
        axes[2].axvline(best_epoch, color='green', ls='--', alpha=0.5)
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('F1-Score')
    axes[2].set_title('Validation F1', fontweight='bold'); axes[2].grid(alpha=0.3)

    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=120, bbox_inches='tight')
    plt.close(fig)
    print(f'[SAVED] {OUTPUT_DIR / "training_curves.png"}')
else:
    print('Only 1 epoch trained. Training curves skipped.')

[SAVED] C:\Users\s-amm\Downloads\ChestX-ray14\Team505_phase2\outputs\Abdelrahman_Mostafa\training_curves.png


---
## 16–18 — Evaluation on Validation Set

In [13]:
checkpoint = torch.load(best_model_path, map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Loaded best model from epoch {checkpoint["epoch"]} (AUC={checkpoint["val_auc"]:.4f})')

val_loss, val_preds_default, val_probs, val_labels = validate(
    model, val_loader, criterion, DEVICE
)

# ---- Threshold tuning (maximize F1 on validation) ----
best_thr = 0.5
best_thr_f1 = 0.0
for thr in np.arange(0.05, 0.95, 0.05):
    preds_at_thr = (val_probs >= thr).astype(int)
    f1_at_thr = f1_score(val_labels, preds_at_thr, zero_division=0)
    if f1_at_thr > best_thr_f1:
        best_thr_f1 = f1_at_thr
        best_thr = round(thr, 2)

print(f'\nOptimal threshold (max F1 on val): {best_thr}')

val_preds_tuned = (val_probs >= best_thr).astype(int)

acc_d  = accuracy_score(val_labels, val_preds_default)
prec_d = precision_score(val_labels, val_preds_default, zero_division=0)
rec_d  = recall_score(val_labels, val_preds_default, zero_division=0)
f1_d   = f1_score(val_labels, val_preds_default, zero_division=0)

acc  = accuracy_score(val_labels, val_preds_tuned)
prec = precision_score(val_labels, val_preds_tuned, zero_division=0)
rec  = recall_score(val_labels, val_preds_tuned, zero_division=0)
f1   = f1_score(val_labels, val_preds_tuned, zero_division=0)
auc  = roc_auc_score(val_labels, val_probs) if val_labels.sum() > 0 else 0.0

print()
print('=' * 60)
print('VALIDATION METRICS (Best Model)')
print('=' * 60)
print(f'                    Default (0.5)    Tuned ({best_thr})')
print(f'  Accuracy  :       {acc_d:.4f}           {acc:.4f}')
print(f'  Precision :       {prec_d:.4f}           {prec:.4f}')
print(f'  Recall    :       {rec_d:.4f}           {rec:.4f}')
print(f'  F1-Score  :       {f1_d:.4f}           {f1:.4f}')
print(f'  ROC-AUC   :       {auc:.4f}           {auc:.4f}  (threshold-independent)')
print('=' * 60)

val_preds = val_preds_tuned

print()
print(f'Classification report (threshold={best_thr}):')
print(classification_report(
    val_labels, val_preds,
    target_names=['Non-Pneumonia', 'Pneumonia'],
    digits=4
))

Loaded best model from epoch 5 (AUC=0.5770)

Optimal threshold (max F1 on val): 0.55

VALIDATION METRICS (Best Model)
                    Default (0.5)    Tuned (0.55)
  Accuracy  :       0.4977           0.6186
  Precision :       0.1082           0.1220
  Recall    :       0.6364           0.5260
  F1-Score  :       0.1849           0.1980
  ROC-AUC   :       0.5770           0.5770  (threshold-independent)

Classification report (threshold=0.55):
               precision    recall  f1-score   support

Non-Pneumonia     0.9309    0.6277    0.7498      1566
    Pneumonia     0.1220    0.5260    0.1980       154

     accuracy                         0.6186      1720
    macro avg     0.5264    0.5768    0.4739      1720
 weighted avg     0.8584    0.6186    0.7004      1720



In [14]:
cm = confusion_matrix(val_labels, val_preds)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Non-Pneumonia', 'Pneumonia'],
    yticklabels=['Non-Pneumonia', 'Pneumonia'],
    ax=ax
)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)
ax.set_title(
    f'Confusion Matrix (ViT-B/16, thr={best_thr})\n'
    f'Acc={acc:.3f}  F1={f1:.3f}  AUC={auc:.3f}',
    fontsize=12, fontweight='bold'
)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.close(fig)
print(f'[SAVED] {OUTPUT_DIR / "confusion_matrix.png"}')

[SAVED] C:\Users\s-amm\Downloads\ChestX-ray14\Team505_phase2\outputs\Abdelrahman_Mostafa\confusion_matrix.png


---
## 19 — Save Outputs

In [15]:
metrics_dict = {
    'model': 'ViT-B/16',
    'mode': RUN_MODE.upper(),
    'best_epoch': best_epoch,
    'num_epochs_trained': len(history['train_loss']),
    'num_epochs_max': NUM_EPOCHS,
    'early_stopped': len(history['train_loss']) < NUM_EPOCHS,
    'train_images': len(df_train),
    'val_images': len(df_val),
    'threshold': best_thr,
    'accuracy': round(acc, 4),
    'precision': round(prec, 4),
    'recall': round(rec, 4),
    'f1_score': round(f1, 4),
    'roc_auc': round(auc, 4),
    'val_loss': round(val_loss, 4),
    'learning_rate': LEARNING_RATE,
}

metrics_df = pd.DataFrame([metrics_dict])
metrics_df.to_csv(OUTPUT_DIR / 'validation_metrics.csv', index=False)
print(f'[SAVED] {OUTPUT_DIR / "validation_metrics.csv"}')

history_df = pd.DataFrame(history)
history_df.index = history_df.index + 1
history_df.index.name = 'epoch'
history_df.to_csv(OUTPUT_DIR / 'training_history.csv')
print(f'[SAVED] {OUTPUT_DIR / "training_history.csv"}')

print(f'\nAll outputs in {OUTPUT_DIR}:')
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f'  {f.name:35s} ({size_kb:>8.1f} KB)')

[SAVED] C:\Users\s-amm\Downloads\ChestX-ray14\Team505_phase2\outputs\Abdelrahman_Mostafa\validation_metrics.csv
[SAVED] C:\Users\s-amm\Downloads\ChestX-ray14\Team505_phase2\outputs\Abdelrahman_Mostafa\training_history.csv

All outputs in C:\Users\s-amm\Downloads\ChestX-ray14\Team505_phase2\outputs\Abdelrahman_Mostafa:
  .gitkeep                            (     0.0 KB)
  best_model.pth                      (1005648.3 KB)
  confusion_matrix.png                (    29.5 KB)
  training_curves.png                 (    81.4 KB)
  training_history.csv                (     0.4 KB)
  validation_metrics.csv              (     0.2 KB)


---
## 20 — Interpretation of Preliminary Results

In [16]:
print('=' * 70)
print('VIT-B/16 BASELINE — FINAL SUMMARY')
print('=' * 70)
print(f'\nRun mode        : {RUN_MODE.upper()}')
print(f'Train samples   : {len(df_train):,}')
print(f'Val samples     : {len(df_val):,}')
print(f'Train pos/neg   : {n_pos_train:,} / {n_neg_train:,}')
val_pos = int(df_val["target_pneumonia"].sum())
val_neg = len(df_val) - val_pos
print(f'Val   pos/neg   : {val_pos:,} / {val_neg:,}')

print(f'\n--- Training Config ---')
print(f'Learning rate   : {LEARNING_RATE}')
print(f'pos_weight      : {pos_weight_value:.2f}')
print(f'Scheduler       : ReduceLROnPlateau (factor=0.5, patience=2)')
print(f'Early stopping  : patience={EARLY_STOP_PATIENCE} ({"triggered" if len(history["train_loss"]) < NUM_EPOCHS else "not triggered"})')
print(f'Epochs trained  : {len(history["train_loss"])} / {NUM_EPOCHS}')

print(f'\n--- GPU Status ---')
model_dev = next(model.parameters()).device
is_gpu = str(model_dev).startswith('cuda')
print(f'GPU detected    : {CUDA_AVAILABLE}')
print(f'Model device    : {model_dev}')
print(f'GPU active      : {is_gpu}')

print(f'\n--- Best Validation Metrics (epoch {best_epoch}, threshold={best_thr}) ---')
print(f'  ROC-AUC     : {auc:.4f}')
print(f'  F1-Score    : {f1:.4f}')
print(f'  Precision   : {prec:.4f}')
print(f'  Recall      : {rec:.4f}')
print(f'  Accuracy    : {acc:.4f}')
print(f'  Threshold   : {best_thr}')

print(f'\nOutput files:')
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        print(f'  {f}')
print('=' * 70)

VIT-B/16 BASELINE — FINAL SUMMARY

Run mode        : DEV
Train samples   : 6,365
Val samples     : 1,720
Train pos/neg   : 588 / 5,777
Val   pos/neg   : 154 / 1,566

--- Training Config ---
Learning rate   : 3e-05
pos_weight      : 9.82
Scheduler       : ReduceLROnPlateau (factor=0.5, patience=2)
Early stopping  : patience=3 (not triggered)
Epochs trained  : 5 / 5

--- GPU Status ---
GPU detected    : True
Model device    : cuda:0
GPU active      : True

--- Best Validation Metrics (epoch 5, threshold=0.55) ---
  ROC-AUC     : 0.5770
  F1-Score    : 0.1980
  Precision   : 0.1220
  Recall      : 0.5260
  Accuracy    : 0.6186
  Threshold   : 0.55

Output files:
  C:\Users\s-amm\Downloads\ChestX-ray14\Team505_phase2\outputs\Abdelrahman_Mostafa\.gitkeep
  C:\Users\s-amm\Downloads\ChestX-ray14\Team505_phase2\outputs\Abdelrahman_Mostafa\best_model.pth
  C:\Users\s-amm\Downloads\ChestX-ray14\Team505_phase2\outputs\Abdelrahman_Mostafa\confusion_matrix.png
  C:\Users\s-amm\Downloads\ChestX-ray1

### Interpretation

**ViT-B/16 preliminary baseline observations:**

- **Weighted loss:** `BCEWithLogitsLoss` with `pos_weight` compensates for the severe class imbalance (~9.2% positive rate in `train_dev`).
- **ROC-AUC** is the primary metric for imbalanced binary classification. Accuracy alone is misleading.
- **Threshold tuning** on validation predictions finds the threshold that maximizes F1-score, which is more appropriate than the default 0.5 for imbalanced data.
- **Early stopping** prevents overfitting by monitoring validation AUC and stopping when performance degrades.
- **Mild augmentation** (rotation ±10°, color jitter) provides data diversity without making radiographs unrealistic.

**For final Phase 2 reporting:** set `RUN_MODE = "full"` to train on the complete dataset.

---
## 21 — Next Steps

1. **Full training:** Set `RUN_MODE = "full"` and train on the complete `train.csv + val.csv` splits.
2. **Compare baselines:** Compare ViT-B/16 results against DenseNet-121, DenseNet-201, and Xception under identical conditions.
3. **Phase 3 — XAI (4 methods planned):** Grad-CAM · LIME · SHAP · Integrated Gradients on ViT-B/16 predictions.
4. **Threshold tuning on full data:** Re-tune the classification threshold after full training.
5. **Final project scope:** Expand to 9–12 total models per team-size requirement; comparative XAI analysis across all models.

---
**End of notebook — Abdelrahman Mostafa, ViT-B/16 Baseline (Singh et al.)**
